
# FairWarn-SHS — Fairness Regularization Refinement and Global λ Selection

This notebook keeps the **same single fairness modification** used in Notebook 07:

\[
L_{\text{FairWarn}}
=
L_{\text{weighted cross-entropy}}
+
\lambda L_{\text{EO}}
\]

where \(L_{\text{EO}}\) is the residence equal-opportunity regularization term.

Notebook 07 showed that the original λ range was too weak and that per-seed λ
selection usually returned λ = 0. Notebook 08 therefore:

1. tests a wider λ range;
2. aggregates validation results across all five seeds;
3. chooses **one global λ** using validation results only;
4. compares that fixed λ with standard GraphSAGE on the paired held-out test sets.

The test set is not used to choose λ.


In [ ]:
!pip -q install torch-geometric pandas numpy scikit-learn matplotlib scipy

In [ ]:

from google.colab import files
uploaded = files.upload()

# Upload:
# FairWarn_SHS_Node_Features.csv
# FairWarn_SHS_Edge_List.csv


In [ ]:

import random
from copy import deepcopy

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F

from scipy.stats import wilcoxon
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score,
    accuracy_score,
    brier_score_loss,
    confusion_matrix,
)
from torch_geometric.data import Data
from torch_geometric.nn import SAGEConv

NODE_FILE = "FairWarn_SHS_Node_Features.csv"
EDGE_FILE = "FairWarn_SHS_Edge_List.csv"

SEEDS = [42, 123, 456, 789, 1010]

# Wider range than Notebook 07.
LAMBDAS = [0.00, 0.10, 0.20, 0.50, 1.00, 2.00, 5.00]

# A lambda remains eligible when its mean validation AUC-PR is
# no more than 0.02 below the best mean validation AUC-PR.
UTILITY_TOLERANCE = 0.02

MAX_EPOCHS = 500
PATIENCE = 40

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)
print("Lambdas:", LAMBDAS)


In [ ]:

nodes = pd.read_csv(NODE_FILE)
edges = pd.read_csv(EDGE_FILE)

labelled_mask_np = (
    nodes["Label_Available"].eq(1) &
    nodes["TARGET_AtRisk"].notna()
).to_numpy()

node_to_index = {
    node_id: index
    for index, node_id in enumerate(nodes["Node_ID"])
}

edge_pairs = []

for _, row in edges.iterrows():
    source_id = row["Source_Node_ID"]
    target_id = row["Target_Node_ID"]

    if (
        source_id in node_to_index
        and target_id in node_to_index
    ):
        s = node_to_index[source_id]
        t = node_to_index[target_id]

        edge_pairs.extend([
            (s, t),
            (t, s),
        ])

edge_index = torch.tensor(
    edge_pairs,
    dtype=torch.long,
).t().contiguous()

excluded_columns = {
    "Node_ID",
    "Roster_Code",
    "School_Code",
    "Class_Code",
    "Label_Available",
    "TARGET_AtRisk",
}

feature_columns = [
    c for c in nodes.columns
    if c not in excluded_columns
]

X_raw = nodes[feature_columns].copy()

numeric_columns = (
    X_raw
    .select_dtypes(include=[np.number])
    .columns
    .tolist()
)

categorical_columns = [
    c for c in X_raw.columns
    if c not in numeric_columns
]

for c in numeric_columns:
    X_raw[c] = X_raw[c].fillna(
        X_raw[c].median()
    )

for c in categorical_columns:
    mode = X_raw[c].mode(dropna=True)
    X_raw[c] = X_raw[c].fillna(
        mode.iloc[0]
        if not mode.empty
        else "Missing"
    )

preprocessor = ColumnTransformer([
    (
        "numeric",
        StandardScaler(),
        numeric_columns,
    ),
    (
        "categorical",
        OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False,
        ),
        categorical_columns,
    ),
])

X = preprocessor.fit_transform(
    X_raw
).astype(np.float32)

y = (
    nodes["TARGET_AtRisk"]
    .fillna(-1)
    .astype(int)
    .to_numpy()
)

residence_text = (
    nodes["Q6_Residence"]
    .fillna("Missing")
    .astype(str)
    .str.strip()
    .str.lower()
)

residence_group = np.full(
    len(nodes),
    -1,
    dtype=np.int64,
)

residence_group[
    residence_text.str.contains(
        "board",
        regex=False,
    )
] = 0

residence_group[
    residence_text.str.contains(
        "day",
        regex=False,
    )
] = 1

graph_data = Data(
    x=torch.tensor(
        X,
        dtype=torch.float32,
    ),
    edge_index=edge_index,
    y=torch.tensor(
        y,
        dtype=torch.long,
    ),
    residence=torch.tensor(
        residence_group,
        dtype=torch.long,
    ),
)

print("Nodes:", graph_data.num_nodes)
print("Labelled nodes:", int(labelled_mask_np.sum()))
print("Undirected edges:", edge_index.shape[1] // 2)
print("Boarding nodes:", int((residence_group == 0).sum()))
print("Day nodes:", int((residence_group == 1).sum()))
print("Encoded feature dimension:", graph_data.num_node_features)


In [ ]:

def make_masks(seed):
    labelled_indices = np.where(
        labelled_mask_np
    )[0]

    labelled_y = y[
        labelled_indices
    ]

    train_validation_indices, test_indices = train_test_split(
        labelled_indices,
        test_size=0.20,
        stratify=labelled_y,
        random_state=seed,
    )

    train_validation_y = y[
        train_validation_indices
    ]

    train_indices, validation_indices = train_test_split(
        train_validation_indices,
        test_size=0.1875,
        stratify=train_validation_y,
        random_state=seed,
    )

    masks = []

    for indices in [
        train_indices,
        validation_indices,
        test_indices,
    ]:
        mask = torch.zeros(
            len(y),
            dtype=torch.bool,
        )
        mask[indices] = True
        masks.append(mask)

    return masks


class FairWarnGraphSAGE(torch.nn.Module):
    def __init__(self, in_channels):
        super().__init__()

        self.conv1 = SAGEConv(
            in_channels,
            64,
            aggr="mean",
        )

        self.conv2 = SAGEConv(
            64,
            32,
            aggr="mean",
        )

        self.classifier = torch.nn.Linear(
            32,
            2,
        )

        self.dropout = 0.35

    def forward(self, x, edge_index):
        x = self.conv1(
            x,
            edge_index,
        )
        x = F.relu(x)
        x = F.dropout(
            x,
            p=self.dropout,
            training=self.training,
        )

        x = self.conv2(
            x,
            edge_index,
        )
        x = F.relu(x)
        x = F.dropout(
            x,
            p=self.dropout,
            training=self.training,
        )

        return self.classifier(x)


In [ ]:

def soft_equal_opportunity_gap(
    positive_probability,
    labels,
    residence,
    mask,
):
    positive_mask = (
        mask
        & labels.eq(1)
        & residence.ge(0)
    )

    boarding_positive = (
        positive_mask
        & residence.eq(0)
    )

    day_positive = (
        positive_mask
        & residence.eq(1)
    )

    if (
        boarding_positive.sum() == 0
        or day_positive.sum() == 0
    ):
        return torch.tensor(
            0.0,
            device=positive_probability.device,
        )

    boarding_mean = positive_probability[
        boarding_positive
    ].mean()

    day_mean = positive_probability[
        day_positive
    ].mean()

    return torch.abs(
        boarding_mean - day_mean
    )


def hard_residence_metrics(
    true_labels,
    probabilities,
    predictions,
    residence_values,
):
    group_rows = []

    for code, name in {
        0: "Boarding",
        1: "Day",
    }.items():

        mask = residence_values == code

        group_true = true_labels[mask]
        group_prob = probabilities[mask]
        group_pred = predictions[mask]

        if len(group_true) == 0:
            continue

        tn, fp, fn, tp = confusion_matrix(
            group_true,
            group_pred,
            labels=[0, 1],
        ).ravel()

        recall = recall_score(
            group_true,
            group_pred,
            zero_division=0,
        )

        fpr = (
            fp / (fp + tn)
            if (fp + tn) > 0
            else np.nan
        )

        group_rows.append({
            "Residence": name,
            "N": len(group_true),
            "AtRisk_N": int(
                np.sum(group_true == 1)
            ),
            "NotAtRisk_N": int(
                np.sum(group_true == 0)
            ),
            "Selection_Rate": float(
                np.mean(group_pred == 1)
            ),
            "Precision_AtRisk": precision_score(
                group_true,
                group_pred,
                zero_division=0,
            ),
            "Recall_AtRisk": recall,
            "F1_AtRisk": f1_score(
                group_true,
                group_pred,
                zero_division=0,
            ),
            "False_Positive_Rate": fpr,
            "TP": int(tp),
            "FP": int(fp),
            "TN": int(tn),
            "FN": int(fn),
        })

    group_df = pd.DataFrame(
        group_rows
    )

    if len(group_df) < 2:
        return group_df, {
            "Residence_Demographic_Parity_Gap": np.nan,
            "Residence_Equal_Opportunity_Gap": np.nan,
            "Residence_FPR_Gap": np.nan,
            "Residence_Equalized_Odds_Gap": np.nan,
            "Residence_F1_Gap": np.nan,
        }

    dp_gap = (
        group_df["Selection_Rate"].max()
        - group_df["Selection_Rate"].min()
    )

    eo_gap = (
        group_df["Recall_AtRisk"].max()
        - group_df["Recall_AtRisk"].min()
    )

    fpr_gap = (
        group_df["False_Positive_Rate"].max()
        - group_df["False_Positive_Rate"].min()
    )

    f1_gap = (
        group_df["F1_AtRisk"].max()
        - group_df["F1_AtRisk"].min()
    )

    return group_df, {
        "Residence_Demographic_Parity_Gap": dp_gap,
        "Residence_Equal_Opportunity_Gap": eo_gap,
        "Residence_FPR_Gap": fpr_gap,
        "Residence_Equalized_Odds_Gap": max(
            eo_gap,
            fpr_gap,
        ),
        "Residence_F1_Gap": f1_gap,
    }


def overall_metrics(
    true_labels,
    probabilities,
    predictions,
):
    return {
        "AUC_ROC": roc_auc_score(
            true_labels,
            probabilities,
        ),
        "AUC_PR": average_precision_score(
            true_labels,
            probabilities,
        ),
        "Precision_AtRisk": precision_score(
            true_labels,
            predictions,
            zero_division=0,
        ),
        "Recall_AtRisk": recall_score(
            true_labels,
            predictions,
            zero_division=0,
        ),
        "F1_AtRisk": f1_score(
            true_labels,
            predictions,
            zero_division=0,
        ),
        "Weighted_F1": f1_score(
            true_labels,
            predictions,
            average="weighted",
            zero_division=0,
        ),
        "Balanced_Accuracy": balanced_accuracy_score(
            true_labels,
            predictions,
        ),
        "Accuracy": accuracy_score(
            true_labels,
            predictions,
        ),
        "Brier_Score": brier_score_loss(
            true_labels,
            probabilities,
        ),
    }


In [ ]:

def train_seed_lambda(
    seed,
    fairness_lambda,
):
    set_seed(seed)

    train_mask, validation_mask, test_mask = make_masks(
        seed
    )

    graph = graph_data.clone()

    graph.train_mask = train_mask
    graph.validation_mask = validation_mask
    graph.test_mask = test_mask

    graph = graph.to(device)

    model = FairWarnGraphSAGE(
        graph.num_node_features
    ).to(device)

    train_labels = graph.y[
        graph.train_mask
    ]

    class_counts = torch.bincount(
        train_labels,
        minlength=2,
    ).float()

    class_weights = (
        class_counts.sum()
        / (
            2.0
            * class_counts.clamp_min(1.0)
        )
    ).to(device)

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=0.005,
        weight_decay=5e-4,
    )

    best_state = None
    best_epoch = 0
    best_validation_auc_pr = -np.inf
    best_validation_soft_eo_gap = np.inf
    wait = 0

    history_rows = []

    for epoch in range(
        1,
        MAX_EPOCHS + 1,
    ):
        model.train()
        optimizer.zero_grad()

        logits = model(
            graph.x,
            graph.edge_index,
        )

        probability = torch.softmax(
            logits,
            dim=1,
        )[:, 1]

        prediction_loss = F.cross_entropy(
            logits[graph.train_mask],
            graph.y[graph.train_mask],
            weight=class_weights,
        )

        fairness_loss = soft_equal_opportunity_gap(
            probability,
            graph.y,
            graph.residence,
            graph.train_mask,
        )

        total_loss = (
            prediction_loss
            + fairness_lambda
            * fairness_loss
        )

        total_loss.backward()
        optimizer.step()

        model.eval()

        with torch.no_grad():
            validation_logits = model(
                graph.x,
                graph.edge_index,
            )

            validation_probability_all = torch.softmax(
                validation_logits,
                dim=1,
            )[:, 1]

            validation_true = (
                graph.y[
                    graph.validation_mask
                ]
                .cpu()
                .numpy()
            )

            validation_probability = (
                validation_probability_all[
                    graph.validation_mask
                ]
                .cpu()
                .numpy()
            )

            validation_auc_pr = average_precision_score(
                validation_true,
                validation_probability,
            )

            validation_soft_eo_gap = float(
                soft_equal_opportunity_gap(
                    validation_probability_all,
                    graph.y,
                    graph.residence,
                    graph.validation_mask,
                )
                .cpu()
                .item()
            )

        history_rows.append({
            "Seed": seed,
            "Lambda": fairness_lambda,
            "Epoch": epoch,
            "Prediction_Loss": float(
                prediction_loss.item()
            ),
            "Fairness_Loss": float(
                fairness_loss.item()
            ),
            "Total_Loss": float(
                total_loss.item()
            ),
            "Validation_AUC_PR": float(
                validation_auc_pr
            ),
            "Validation_Soft_EO_Gap": float(
                validation_soft_eo_gap
            ),
        })

        # Early stopping remains based on validation AUC-PR.
        # This keeps model fitting separate from global lambda selection.
        if (
            validation_auc_pr
            > best_validation_auc_pr + 1e-6
        ):
            best_validation_auc_pr = validation_auc_pr
            best_validation_soft_eo_gap = (
                validation_soft_eo_gap
            )
            best_epoch = epoch
            best_state = deepcopy(
                model.state_dict()
            )
            wait = 0
        else:
            wait += 1

        if wait >= PATIENCE:
            break

    model.load_state_dict(
        best_state
    )
    model.eval()

    with torch.no_grad():
        logits = model(
            graph.x,
            graph.edge_index,
        )

        probability_all = torch.softmax(
            logits,
            dim=1,
        )[:, 1]

        prediction_all = torch.argmax(
            logits,
            dim=1,
        )

    test_indices = (
        torch.where(
            graph.test_mask
        )[0]
        .cpu()
        .numpy()
    )

    test_true = (
        graph.y[
            graph.test_mask
        ]
        .cpu()
        .numpy()
    )

    test_probability = (
        probability_all[
            graph.test_mask
        ]
        .cpu()
        .numpy()
    )

    test_prediction = (
        prediction_all[
            graph.test_mask
        ]
        .cpu()
        .numpy()
    )

    test_residence = (
        graph.residence[
            graph.test_mask
        ]
        .cpu()
        .numpy()
    )

    test_soft_eo_gap = float(
        soft_equal_opportunity_gap(
            probability_all,
            graph.y,
            graph.residence,
            graph.test_mask,
        )
        .cpu()
        .item()
    )

    overall = overall_metrics(
        test_true,
        test_probability,
        test_prediction,
    )

    residence_df, hard_gaps = hard_residence_metrics(
        test_true,
        test_probability,
        test_prediction,
        test_residence,
    )

    result = {
        "Seed": seed,
        "Lambda": fairness_lambda,
        "Best_Epoch": best_epoch,
        "Validation_AUC_PR": best_validation_auc_pr,
        "Validation_Soft_EO_Gap": best_validation_soft_eo_gap,
        "Test_Soft_EO_Gap": test_soft_eo_gap,
        **overall,
        **hard_gaps,
    }

    prediction_df = pd.DataFrame({
        "Seed": seed,
        "Lambda": fairness_lambda,
        "Node_Index": test_indices,
        "Node_ID": (
            nodes.iloc[
                test_indices
            ]["Node_ID"]
            .to_numpy()
        ),
        "True_Label": test_true,
        "Predicted_Label": test_prediction,
        "AtRisk_Probability": test_probability,
        "Residence_Code": test_residence,
        "Residence": np.where(
            test_residence == 0,
            "Boarding",
            np.where(
                test_residence == 1,
                "Day",
                "Unmapped",
            ),
        ),
    })

    residence_df["Seed"] = seed
    residence_df["Lambda"] = fairness_lambda

    history_df = pd.DataFrame(
        history_rows
    )

    return {
        "result": result,
        "predictions": prediction_df,
        "residence_metrics": residence_df,
        "history": history_df,
    }



## Stage 1 — Wider λ experiment

This runs:

```text
7 λ values × 5 seeds = 35 models
```

Do not select λ from the test metrics printed during this stage.


In [ ]:

run_store = {}
result_rows = []
prediction_tables = []
residence_tables = []
history_tables = []

for seed in SEEDS:
    run_store[seed] = {}

    for fairness_lambda in LAMBDAS:
        print(
            f"Training seed={seed}, "
            f"lambda={fairness_lambda:.2f}"
        )

        output = train_seed_lambda(
            seed,
            fairness_lambda,
        )

        run_store[seed][
            fairness_lambda
        ] = output

        result_rows.append(
            output["result"]
        )

        prediction_tables.append(
            output["predictions"]
        )

        residence_tables.append(
            output["residence_metrics"]
        )

        history_tables.append(
            output["history"]
        )

        r = output["result"]

        print(
            f"  Validation AUC-PR="
            f"{r['Validation_AUC_PR']:.4f} | "
            f"Validation soft EO="
            f"{r['Validation_Soft_EO_Gap']:.4f}"
        )

results_df = pd.DataFrame(
    result_rows
)

all_predictions_df = pd.concat(
    prediction_tables,
    ignore_index=True,
)

all_residence_metrics_df = pd.concat(
    residence_tables,
    ignore_index=True,
)

all_history_df = pd.concat(
    history_tables,
    ignore_index=True,
)



## Stage 2 — Choose one global λ using validation results only

Selection rule:

1. calculate mean validation AUC-PR and mean validation soft EO gap for every λ;
2. find the best mean validation AUC-PR;
3. retain λ values within 0.02 AUC-PR of that best value;
4. among those eligible values, choose the λ with the smallest mean validation
   soft EO gap;
5. use the smaller λ to break an exact tie.


In [ ]:

validation_summary_rows = []

for fairness_lambda, group in results_df.groupby(
    "Lambda"
):
    validation_summary_rows.append({
        "Lambda": fairness_lambda,
        "Seeds": group["Seed"].nunique(),
        "Validation_AUC_PR_Mean": (
            group["Validation_AUC_PR"].mean()
        ),
        "Validation_AUC_PR_SD": (
            group["Validation_AUC_PR"].std(
                ddof=1
            )
        ),
        "Validation_Soft_EO_Gap_Mean": (
            group[
                "Validation_Soft_EO_Gap"
            ].mean()
        ),
        "Validation_Soft_EO_Gap_SD": (
            group[
                "Validation_Soft_EO_Gap"
            ].std(ddof=1)
        ),
    })

validation_summary_df = pd.DataFrame(
    validation_summary_rows
).sort_values("Lambda")

best_validation_auc_pr = (
    validation_summary_df[
        "Validation_AUC_PR_Mean"
    ].max()
)

eligible_validation_df = (
    validation_summary_df[
        validation_summary_df[
            "Validation_AUC_PR_Mean"
        ]
        >= (
            best_validation_auc_pr
            - UTILITY_TOLERANCE
        )
    ]
    .copy()
)

selected_global_row = (
    eligible_validation_df
    .sort_values(
        [
            "Validation_Soft_EO_Gap_Mean",
            "Lambda",
        ],
        ascending=[
            True,
            True,
        ],
    )
    .iloc[0]
)

SELECTED_GLOBAL_LAMBDA = float(
    selected_global_row["Lambda"]
)

print(
    "Best mean validation AUC-PR:",
    round(
        best_validation_auc_pr,
        6,
    )
)

print(
    "Selected global lambda:",
    SELECTED_GLOBAL_LAMBDA
)

display(
    validation_summary_df
)

display(
    eligible_validation_df
)


In [ ]:

# Test-set comparison is opened only after global lambda selection.
comparison_rows = []
comparison_predictions = []
comparison_residence_rows = []

for seed in SEEDS:
    standard_output = (
        run_store[seed][0.00]
    )

    fair_output = (
        run_store[seed][
            SELECTED_GLOBAL_LAMBDA
        ]
    )

    standard_result = {
        **standard_output["result"],
        "Model": "Standard GraphSAGE",
        "Fixed_Global_Lambda": 0.00,
    }

    fair_result = {
        **fair_output["result"],
        "Model": "FairWarn-SHS",
        "Fixed_Global_Lambda": (
            SELECTED_GLOBAL_LAMBDA
        ),
    }

    comparison_rows.extend([
        standard_result,
        fair_result,
    ])

    standard_predictions = (
        standard_output[
            "predictions"
        ].copy()
    )
    standard_predictions["Model"] = (
        "Standard GraphSAGE"
    )

    fair_predictions = (
        fair_output[
            "predictions"
        ].copy()
    )
    fair_predictions["Model"] = (
        "FairWarn-SHS"
    )

    comparison_predictions.extend([
        standard_predictions,
        fair_predictions,
    ])

    standard_residence = (
        standard_output[
            "residence_metrics"
        ].copy()
    )
    standard_residence["Model"] = (
        "Standard GraphSAGE"
    )

    fair_residence = (
        fair_output[
            "residence_metrics"
        ].copy()
    )
    fair_residence["Model"] = (
        "FairWarn-SHS"
    )

    comparison_residence_rows.extend([
        standard_residence,
        fair_residence,
    ])

comparison_seed_df = pd.DataFrame(
    comparison_rows
)

comparison_predictions_df = pd.concat(
    comparison_predictions,
    ignore_index=True,
)

comparison_residence_df = pd.concat(
    comparison_residence_rows,
    ignore_index=True,
)

comparison_seed_df[
    [
        "Model",
        "Seed",
        "Fixed_Global_Lambda",
        "AUC_PR",
        "Recall_AtRisk",
        "F1_AtRisk",
        "Residence_Equal_Opportunity_Gap",
        "Residence_Equalized_Odds_Gap",
        "Test_Soft_EO_Gap",
    ]
].sort_values(
    ["Seed", "Model"]
)


In [ ]:

summary_metrics = [
    "AUC_ROC",
    "AUC_PR",
    "Precision_AtRisk",
    "Recall_AtRisk",
    "F1_AtRisk",
    "Weighted_F1",
    "Balanced_Accuracy",
    "Accuracy",
    "Brier_Score",
    "Test_Soft_EO_Gap",
    "Residence_Demographic_Parity_Gap",
    "Residence_Equal_Opportunity_Gap",
    "Residence_FPR_Gap",
    "Residence_Equalized_Odds_Gap",
    "Residence_F1_Gap",
]

summary_rows = []

for model_name, group in comparison_seed_df.groupby(
    "Model"
):
    row = {
        "Model": model_name,
        "Seeds": group["Seed"].nunique(),
        "Fixed_Global_Lambda": (
            group[
                "Fixed_Global_Lambda"
            ].iloc[0]
        ),
    }

    for metric in summary_metrics:
        row[
            f"{metric}_Mean"
        ] = group[metric].mean()

        row[
            f"{metric}_SD"
        ] = group[metric].std(
            ddof=1
        )

    summary_rows.append(row)

final_summary_df = pd.DataFrame(
    summary_rows
)

final_summary_df


In [ ]:

# Fixed-lambda test summary for transparency.
lambda_test_summary_rows = []

for fairness_lambda, group in results_df.groupby(
    "Lambda"
):
    row = {
        "Lambda": fairness_lambda,
        "Seeds": group["Seed"].nunique(),
    }

    for metric in summary_metrics:
        row[
            f"{metric}_Mean"
        ] = group[metric].mean()

        row[
            f"{metric}_SD"
        ] = group[metric].std(
            ddof=1
        )

    lambda_test_summary_rows.append(
        row
    )

lambda_test_summary_df = (
    pd.DataFrame(
        lambda_test_summary_rows
    )
    .sort_values("Lambda")
)

lambda_test_summary_df[
    [
        "Lambda",
        "AUC_PR_Mean",
        "Recall_AtRisk_Mean",
        "F1_AtRisk_Mean",
        "Residence_Equal_Opportunity_Gap_Mean",
        "Residence_Equalized_Odds_Gap_Mean",
        "Test_Soft_EO_Gap_Mean",
    ]
]


In [ ]:

# Paired Wilcoxon tests across seeds.
test_metrics = [
    "AUC_PR",
    "Recall_AtRisk",
    "F1_AtRisk",
    "Balanced_Accuracy",
    "Residence_Equal_Opportunity_Gap",
    "Residence_Equalized_Odds_Gap",
    "Test_Soft_EO_Gap",
]

statistical_rows = []

for metric in test_metrics:
    standard = (
        comparison_seed_df[
            comparison_seed_df[
                "Model"
            ].eq(
                "Standard GraphSAGE"
            )
        ]
        .sort_values("Seed")[
            metric
        ]
        .to_numpy()
    )

    fair = (
        comparison_seed_df[
            comparison_seed_df[
                "Model"
            ].eq(
                "FairWarn-SHS"
            )
        ]
        .sort_values("Seed")[
            metric
        ]
        .to_numpy()
    )

    differences = (
        fair - standard
    )

    try:
        statistic, p_value = wilcoxon(
            fair,
            standard,
            zero_method="wilcox",
            alternative="two-sided",
        )
    except ValueError:
        statistic = np.nan
        p_value = np.nan

    nonzero = differences[
        differences != 0
    ]

    if len(nonzero) == 0:
        rank_biserial = 0.0
    else:
        ranks = (
            pd.Series(
                np.abs(nonzero)
            )
            .rank(
                method="average"
            )
            .to_numpy()
        )

        positive_rank_sum = ranks[
            nonzero > 0
        ].sum()

        negative_rank_sum = ranks[
            nonzero < 0
        ].sum()

        rank_biserial = (
            positive_rank_sum
            - negative_rank_sum
        ) / (
            positive_rank_sum
            + negative_rank_sum
        )

    statistical_rows.append({
        "Metric": metric,
        "Standard_Mean": standard.mean(),
        "FairWarn_Mean": fair.mean(),
        "Mean_Difference_FairWarn_Minus_Standard": (
            differences.mean()
        ),
        "Wilcoxon_Statistic": statistic,
        "P_Value_Two_Sided": p_value,
        "Rank_Biserial_Effect_Size": rank_biserial,
        "Pairs": len(differences),
    })

statistical_tests_df = pd.DataFrame(
    statistical_rows
)

statistical_tests_df


In [ ]:

# Validation utility-fairness chart.
plt.figure(figsize=(8, 5))

plt.plot(
    validation_summary_df[
        "Validation_Soft_EO_Gap_Mean"
    ],
    validation_summary_df[
        "Validation_AUC_PR_Mean"
    ],
    marker="o",
)

for _, row in validation_summary_df.iterrows():
    plt.annotate(
        f"λ={row['Lambda']:.2f}",
        (
            row[
                "Validation_Soft_EO_Gap_Mean"
            ],
            row[
                "Validation_AUC_PR_Mean"
            ],
        ),
        xytext=(5, 5),
        textcoords="offset points",
    )

plt.xlabel(
    "Mean validation soft EO gap"
)
plt.ylabel(
    "Mean validation AUC-PR"
)
plt.title(
    "Global λ selection from validation results"
)
plt.tight_layout()
plt.show()


In [ ]:

# Residence recall after final global-lambda selection.
recall_summary = (
    comparison_residence_df
    .groupby(
        ["Model", "Residence"]
    )["Recall_AtRisk"]
    .agg(["mean", "std"])
    .reset_index()
)

display(
    recall_summary
)

models = [
    "Standard GraphSAGE",
    "FairWarn-SHS",
]

residences = [
    "Boarding",
    "Day",
]

x = np.arange(
    len(residences)
)

width = 0.35

plt.figure(figsize=(8, 5))

for i, model_name in enumerate(
    models
):
    temp = (
        recall_summary[
            recall_summary[
                "Model"
            ].eq(model_name)
        ]
        .set_index("Residence")
        .reindex(residences)
    )

    offset = (
        i - 0.5
    ) * width

    plt.bar(
        x + offset,
        temp["mean"],
        width,
        yerr=temp["std"],
        capsize=4,
        label=model_name,
    )

plt.xticks(
    x,
    residences,
)
plt.ylabel(
    "At-risk recall"
)
plt.title(
    "Residence recall: standard vs final FairWarn-SHS"
)
plt.legend()
plt.tight_layout()
plt.show()



## Decision rule after this notebook

The selected global λ is retained only if the results show a meaningful fairness
improvement with an acceptable predictive-utility trade-off.

If the wider λ range still fails to improve residence fairness reliably, the study
must report that the tested fairness regularizer did not provide a beneficial
trade-off. The result must not be altered merely to force a positive outcome.


In [ ]:

validation_summary_df.to_csv(
    "fairwarn08_validation_lambda_summary.csv",
    index=False,
)

eligible_validation_df.to_csv(
    "fairwarn08_validation_eligible_lambdas.csv",
    index=False,
)

pd.DataFrame([{
    "Selected_Global_Lambda": SELECTED_GLOBAL_LAMBDA,
    "Utility_Tolerance": UTILITY_TOLERANCE,
    "Best_Validation_AUC_PR_Mean": best_validation_auc_pr,
    "Selected_Validation_AUC_PR_Mean": selected_global_row[
        "Validation_AUC_PR_Mean"
    ],
    "Selected_Validation_Soft_EO_Gap_Mean": selected_global_row[
        "Validation_Soft_EO_Gap_Mean"
    ],
}]).to_csv(
    "fairwarn08_selected_global_lambda.csv",
    index=False,
)

results_df.to_csv(
    "fairwarn08_all_lambda_seed_metrics.csv",
    index=False,
)

lambda_test_summary_df.to_csv(
    "fairwarn08_lambda_test_summary_mean_sd.csv",
    index=False,
)

comparison_seed_df.to_csv(
    "fairwarn08_standard_vs_final_seed_metrics.csv",
    index=False,
)

final_summary_df.to_csv(
    "fairwarn08_standard_vs_final_summary_mean_sd.csv",
    index=False,
)

comparison_residence_df.to_csv(
    "fairwarn08_residence_group_metrics.csv",
    index=False,
)

statistical_tests_df.to_csv(
    "fairwarn08_paired_statistical_tests.csv",
    index=False,
)

comparison_predictions_df.to_csv(
    "fairwarn08_standard_vs_final_predictions.csv",
    index=False,
)

all_history_df.to_csv(
    "fairwarn08_training_history.csv",
    index=False,
)

files.download(
    "fairwarn08_selected_global_lambda.csv"
)

files.download(
    "fairwarn08_validation_lambda_summary.csv"
)

files.download(
    "fairwarn08_standard_vs_final_summary_mean_sd.csv"
)

files.download(
    "fairwarn08_residence_group_metrics.csv"
)

files.download(
    "fairwarn08_paired_statistical_tests.csv"
)

files.download(
    "fairwarn08_standard_vs_final_seed_metrics.csv"
)

files.download(
    "fairwarn08_lambda_test_summary_mean_sd.csv"
)

files.download(
    "fairwarn08_standard_vs_final_predictions.csv"
)

files.download(
    "fairwarn08_all_lambda_seed_metrics.csv"
)
